In [1]:
import tkinter as tk, threading
import imageio
from PIL import Image, ImageTk
from functools import partial
import pandas as pd
import numpy as np
from os import listdir, path, mkdir

In [2]:
import sys
import trace
import time
import winsound
import datetime

In [3]:
lst = []
p_number = 5
category = "basketball" #todo add timestamp
folder = f"./artifacts output/{category}"
videos_folders_list = [f"{folder}/{x}" for x in listdir(folder)]

### Run

In [ ]:
# !pip3 install tkvideoplayer==2.1

In [4]:
# !pip3 install openpyxl

In [5]:
reached_end = False

In [6]:

class thread_with_trace(threading.Thread):
    def __init__(self, *args, **keywords):
        threading.Thread.__init__(self, *args, **keywords)
        self.killed = False
 
    def start(self):
        self.__run_backup = self.run
        self.run = self.__run      
        threading.Thread.start(self)
 
    def __run(self):
        sys.settrace(self.globaltrace)
        self.__run_backup()
        self.run = self.__run_backup
        
 
    def globaltrace(self, frame, event, arg):
        if event == 'call':
            return self.localtrace
        else:
            return None
        
    def localtrace(self, frame, event, arg):
        if self.killed:
            if event == 'line':
                raise SystemExit()
        return self.localtrace
    
    def kill(self):
        self.killed = True


In [7]:
def center_label(window, label):
    # Get the window size
    window.update_idletasks()  # Update "requested size" from geometry manager
    window_width = window.winfo_width()
    window_height = window.winfo_height()

    # Calculate the center of the window
    center_x = window_width // 2
    center_y = window_height // 2

    # Position the label in the center
    label.place(x=center_x, y=center_y, anchor="center")


def center_label_and_buttons(window, label, buttons):
    # Get the window size
    window.update_idletasks()  # Update "requested size" from geometry manager
    window_width = window.winfo_width()
    window_height = window.winfo_height()

    # Calculate the center of the window
    center_x = window_width // 2
    center_y = window_height // 2

    # Position the label in the center
    label.place(x=center_x, y=center_y, anchor="center")

    # Calculate button positions
    button_spacing = 10
    button_width = buttons[0].winfo_reqwidth()
    total_width = (button_width * len(buttons)) + (button_spacing * (len(buttons) - 1))
    start_x = center_x - (total_width // 2)

    for i, button in enumerate(buttons):
        button_x = start_x + i * (button_width + button_spacing)
        button.place(x=button_x, y=center_y + 30, anchor="n")

In [ ]:
def save_result(e, root, video_index, index, thread, folder, video_name, timee, reached_end, lenn):
   row = folder, video_name.split("/")[-1]

   if data.isna().loc[row, "elapsed"]:
      # pass
      data.loc[row, "elapsed"] = time.time() - timee

   else:
      x = data.loc[row, "elapsed"]
      data.loc[row, "elapsed"] = (time.time() - timee) + x

   if e.keysym == "Escape":
      close(root, thread)
   
   if e.keysym == "Up":
      restart(root, video_index, index, thread, reached_end)
   
   if e.keysym == "Right":
      next_video(root, video_index, index, thread, reached_end, lenn)
      
   if e.keysym == "Left":
      previous_video(root, video_index, index, thread, reached_end)
      
   if e.keysym == "space" and reached_end:
         name = folder.split("/")[-1]
         
         data.at[row, "chosen"] = 1
         data.loc[folder].to_excel(f"./p{p_number}/p{p_number}_{category}_{name}.xlsx")
         video_index += 1
         reached_end = False
         restart(root, video_index, 0, thread, reached_end)


In [ ]:
def restart(root, video_index, index, thread, reached_end):
    thread.stop()
    root.destroy()
    open_window(video_index, index, reached_end)

def close(root, thread):
    thread.stop()
    root.destroy()
    

def next_video(root, video_index, index, thread, reached_end, lenn):
    if index == lenn-1:
        return
      
    thread.stop()
    root.destroy()
    index += 1
    open_window(video_index, index, reached_end)

def previous_video(root, video_index, index, thread, reached_end):
    if index == 0:
        return
    
    thread.stop()
    root.destroy()
    index -= 1
    open_window(video_index, index, reached_end)
    
def close(root, thread):
    thread.stop()
    root.destroy()

def open_window(video_index, index, reached_end):
        
    if video_index == len(videos_folders_list):
        return
    
    videos_pairs_list = []

    _folder = videos_folders_list[video_index]
    if index < len(videos_folders_list):
        videos_pairs_list = [f"{_folder}/{file}" for file in listdir(_folder)]

    lenn = len(videos_pairs_list)

    video_name2 = videos_pairs_list[index] 
    
    root = tk.Tk()
    root.state('zoomed')
    root.update_idletasks()
    width = root.winfo_screenwidth()
    height = root.winfo_screenheight()
    root.geometry(f"{width}x{height}")
    
    root.grid_rowconfigure(0, weight=1)
    root.grid_rowconfigure(2, weight=1)
    root.grid_columnconfigure(0, weight=1)
    root.grid_columnconfigure(4, weight=1)
    root.attributes('-topmost', True)

    my_label = TkinterVideo(master=root, scaled=True)

    my_label.load(video_name2)

    # my_label.grid(row=1, column=2, padx=10, pady=10)  # Adjust padx and pady for spacing
    my_label.pack(expand=True, fill="both")
    
    if index == lenn - 1:
        reached_end = True
        winsound.Beep(500, 200) 

    
    start = time.time()
    root.bind("<KeyPress>",lambda e: save_result(e, root, video_index, index, my_label, _folder ,video_name2.split("/")[-1], start, reached_end, lenn))
    
    # root.after(3, lambda: root.focus_force())
    my_label.play()
    root.mainloop()
    

In [10]:
arrays = []
for folder in videos_folders_list:
    for video in listdir(folder):
        arrays.append((folder, video))

: 

In [11]:

if __name__ == "__main__":    

    data = pd.DataFrame(index=pd.MultiIndex.from_tuples(arrays), columns=["chosen", "cideo length","elapsed","replay"])
    
    open_window(0, 0, False)       
    # date = datetime.datetime.now()
    # with pd.ExcelWriter(f"./p{p_number}/p{p_number}_{category}_{date}.xlsx",engine="openpyxl") as writer:
    #     for video in videos_folders_list:
    #         video_name = video.split("/")[-1]
    #         df = data.loc[video]
    #         df.to_excel(writer, sheet_name=video_name)


Exception ignored in: <function PhotoImage.__del__ at 0x0000022FF9457060>
Traceback (most recent call last):
  File "c:\Users\spirolab\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\ImageTk.py", line 132, in __del__
    name = self.__photo.name
           ^^^^^^^^^^^^
AttributeError: 'PhotoImage' object has no attribute '_PhotoImage__photo'
Exception ignored in: <function PhotoImage.__del__ at 0x0000022FF9457060>
Traceback (most recent call last):
  File "c:\Users\spirolab\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\ImageTk.py", line 132, in __del__
    name = self.__photo.name
           ^^^^^^^^^^^^
AttributeError: 'PhotoImage' object has no attribute '_PhotoImage__photo'
Exception ignored in: <function PhotoImage.__del__ at 0x0000022FF9457060>
Traceback (most recent call last):
  File "c:\Users\spirolab\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\ImageTk.py", line 132, in __del__
    name = self.__photo.name
           ^^^^^^^^^^^^


In [12]:
# date = datetime.datetime.now()
# with pd.ExcelWriter(f"./p{p_number}/p{p_number}_{category}_{date}.xlsx",engine="openpyxl") as writer:
#     for video in videos_folders_list:
#         video_name = video.split("/")[-1]
#         df = data.loc[video]
#         df.to_excel(writer, sheet_name=video_name)

In [13]:


# for video in videos_folders_list:
#     video_name = video.split("/")[-1]
#     df = data.loc[video]
#     df.to_excel(f"./p{p_number}/p{p_number}_{category}_{video_name}.xlsx")
